# Outlier Analysis

**방법**: `-40ms`와 `0ms` 두 점으로 선형 외삽 → `+80ms` 위치 예측  
**지표**: 예측 위치 vs. 실제 target 사이의 유클리드 거리  
**목적**: threshold보다 큰 오차를 가진 데이터 비율 확인 → outlier 제거 기준 결정

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.notebook import tqdm

DATA_DIR   = Path('../data')
TRAIN_DIR  = DATA_DIR / 'train'
LABELS_PATH = DATA_DIR / 'train_labels.csv'

In [6]:
train_files = sorted(list(TRAIN_DIR.glob('TRAIN_*.csv')))
labels_df   = pd.read_csv(LABELS_PATH)
labels_dict = labels_df.set_index('id')[['x', 'y', 'z']].T.to_dict('list')

print(f"Train files : {len(train_files)}")
print(f"Label rows  : {len(labels_df)}")

Train files : 10000
Label rows  : 10000


In [7]:
def extrapolate_80ms(seq: np.ndarray) -> np.ndarray:
    """선형 외삽: velocity = 0ms - (-40ms),  pred = 0ms + 2 * velocity"""
    velocity = seq[-1] - seq[-2]
    return seq[-1] + 2 * velocity

In [8]:
import os
from concurrent.futures import ThreadPoolExecutor

def _load_last2(path):
    """마지막 2행(-40ms, 0ms)만 읽음: skiprows=10, max_rows=2"""
    return np.loadtxt(str(path), delimiter=',', skiprows=10, max_rows=2,
                      usecols=(1, 2, 3), dtype=np.float32)

n_workers = min(32, (os.cpu_count() or 1) * 4)
with ThreadPoolExecutor(max_workers=n_workers) as pool:
    last2_list = list(tqdm(pool.map(_load_last2, train_files),
                           total=len(train_files), desc='Loading (parallel)'))

file_ids = []
errors   = []
for path, seq2 in zip(train_files, last2_list):
    fid = path.stem
    if fid not in labels_dict:
        continue
    target   = np.array(labels_dict[fid], dtype=np.float32)
    velocity = seq2[1] - seq2[0]          # 0ms − (−40ms)
    pred     = seq2[1] + 2 * velocity     # +80ms
    errors.append(float(np.linalg.norm(pred - target)))
    file_ids.append(fid)

errors = np.array(errors, dtype=np.float32)
print(f"Computed {len(errors)} samples")


In [ ]:
print("── Error statistics ──────────────────")
print(f"  Mean   : {errors.mean():.6f} m")
print(f"  Median : {np.median(errors):.6f} m")
print(f"  Std    : {errors.std():.6f} m")
print(f"  90th % : {np.percentile(errors, 90):.6f} m")
print(f"  95th % : {np.percentile(errors, 95):.6f} m")
print(f"  99th % : {np.percentile(errors, 99):.6f} m")
print(f"  Max    : {errors.max():.6f} m")

In [ ]:
# ── threshold 설정 (여기서 변경) ──────────────────────────────────────
THRESHOLD = 0.05  # metres
# ─────────────────────────────────────────────────────────────────────

n_outlier = int((errors > THRESHOLD).sum())
ratio     = n_outlier / len(errors)

print(f"Threshold : {THRESHOLD} m")
print(f"Outliers  : {n_outlier} / {len(errors)}  ({ratio*100:.2f}%)")
print(f"Inliers   : {len(errors) - n_outlier}  ({(1-ratio)*100:.2f}%)")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# --- 히스토그램 ---
ax = axes[0]
ax.hist(errors, bins=100, color='steelblue', edgecolor='white')
ax.axvline(THRESHOLD, color='red', linestyle='--', linewidth=1.5,
           label=f'threshold = {THRESHOLD} m')
ax.set_xlabel('Extrapolation Error (m)')
ax.set_ylabel('Count')
ax.set_title('Error Distribution')
ax.legend()

# --- 누적 분포 ---
ax = axes[1]
ax.hist(errors, bins=200, color='steelblue', edgecolor='none',
        cumulative=True, density=True)
ax.axvline(THRESHOLD, color='red', linestyle='--', linewidth=1.5,
           label=f'threshold = {THRESHOLD} m')
ax.set_xlabel('Extrapolation Error (m)')
ax.set_ylabel('Cumulative Proportion')
ax.set_title('Cumulative Distribution')
ax.legend()

# --- threshold sweep ---
thresholds = np.linspace(0.001, errors.max() * 0.5, 300)
ratios = [(errors > t).mean() for t in thresholds]

ax = axes[2]
ax.plot(thresholds, ratios, color='steelblue')
ax.axvline(THRESHOLD, color='red', linestyle='--', linewidth=1.5,
           label=f'threshold = {THRESHOLD}')
ax.set_xlabel('Threshold (m)')
ax.set_ylabel('Outlier Ratio')
ax.set_title('Outlier Ratio vs. Threshold')
ax.grid(True, alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# outlier ID 목록 확인
outlier_ids = [fid for fid, e in zip(file_ids, errors) if e > THRESHOLD]
print(f"Outlier file IDs (first 20): {outlier_ids[:20]}")